# 21 · Confirming a 1h trendline break

A 1h line gives way on a 5m candle. That candle is **rs**. Before the trade is taken, a later
candle has to prove the move held:

* within **`TL_CONFIRM_N`** candles of rs, one must close **beyond rs** — below its low for a
  short, above its high for a long — and be the trade's own colour;
* if instead a candle **closes back over rs**, the setup is cancelled (`TL_CONFIRM_CANCEL`) — a
  bounce voids it rather than leaving it armed.

Set per line timeframe, so 1h breaks must prove themselves while 15m and 5m are taken as they come.

Everything else matches notebook 20, and the run below is repeated with the confirmation OFF, so
the only difference between the two columns is this rule.

Settings: **`nb21_config.py`**.

## 1 · Settings

In [1]:
import sys, os, importlib, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd
from quant.viz import AS_OF_JS

for _p in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.abspath("..")):
    if os.path.isfile(os.path.join(_p, "nb21_config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
import nb21_config as CFG
importlib.reload(CFG)

print(f"{CFG.SYMBOL} {CFG.TF}  {CFG.VIEW_START} -> {CFG.AS_OF or CFG.END}")
print(f"lines fitted and traded on {CFG.TF}+{'+'.join(CFG.LINE_TFS)}")
print(f"confirmation: {CFG.TL_CONFIRM_N}  "
      f"{'all n' if CFG.TL_CONFIRM_ALL else 'one of n'} · {CFG.TL_CONFIRM_MODE} beyond rs · "
      f"{'own colour' if CFG.TL_CONFIRM_COLOUR else 'any colour'} · "
      f"{'cancels on a close back over rs' if CFG.TL_CONFIRM_CANCEL else 'no cancel'}")
print(f"settings from {CFG.__file__}")

BTCUSDT 5m  2026-06-01 -> 2026-06-10
lines fitted and traded on 5m+15m+1h
confirmation: {'1h': 3, '*': 0}  one of n · close beyond rs · own colour · cancels on a close back over rs
settings from C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\nb21_config.py


## 2 · With the confirmation, and without

`variant()` re-runs the identical study with the rule switched off. Nothing else changes, so every
difference below belongs to the confirmation.

In [2]:
from quant.research import fvg_lab, variant, compare_runs, trade_diff

on  = fvg_lab(CFG)
off = fvg_lab(variant(CFG, TL_CONFIRM_N=0, TL_CONFIRM_CANCEL=False))

compare_runs(confirmation_off=off, confirmation_on=on)

[04:07:02] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[04:07:03] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[04:07:03] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[04:07:03] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[04:07:03] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[04:07:03] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[04:07:04] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load
[04:07:06] INFO | quant.engine | backtest done | bars=11,521 trades=3 final_cash=9670.60 return=-3.29% dd=3.29% | 48.7 ms
[04:07:06] INFO | quant.engine | backtest done | bars=11,521 trades=5 final_cash=11015.44 return=10.15% dd=1.72% | 0.6 ms
[04:07:06] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[04:07:07] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[04:07:07] INFO | quant.data | cache HIT  binance BT

,run,trades,total R,wins,R / trade
0,confirmation off,8,6.99,4,0.874
1,confirmation on,8,6.99,4,0.874


## 3 · Which trades it changed

A confirmation can only move an entry later or remove it, so read the rows that appear in one
column and not the other. `1h` in `trigger` is the timeframe this rule is set for.

In [3]:
trade_diff(off=off, on=on)

,side off,trigger off,R off,close_reason off,side on,trigger on,R on,close_reason on
entry,,,,,,,,
2026-06-01 01:40,short,15m,2.802544,take_profit,short,15m,2.802544,take_profit
2026-06-02 07:00,short,15m,2.871383,take_profit,short,15m,2.871383,take_profit
2026-06-05 01:00,short,5m,2.886505,take_profit,short,5m,2.886505,take_profit
2026-06-05 18:50,short,1h,-1.053178,stop_loss,NaN,NaN,NaN,NaN
2026-06-05 19:15,NaN,NaN,NaN,NaN,short,1h,-1.054131,stop_loss
2026-06-05 22:50,long,15m,-1.135041,stop_loss,long,15m,-1.135041,stop_loss
2026-06-06 08:25,long,15m,-1.140345,stop_loss,long,15m,-1.140345,stop_loss
2026-06-07 22:10,long,15m,-1.055388,stop_loss,long,15m,-1.055388,stop_loss
2026-06-09 09:30,short,15m,2.818152,take_profit,short,15m,2.818152,take_profit


## 4 · The chart

In [4]:
fig = on.figure()
# fig

In [5]:
from quant.viz import open_in_browser

open_in_browser(fig, "confirm.html", extra_js=AS_OF_JS)

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/confirm.html')

---
### What to change

| want | setting |
|---|---|
| more or less grace after rs | `TL_CONFIRM_N = {"1h": 3, "*": 0}` |
| every candle must hold, not just one | `TL_CONFIRM_ALL = True` |
| the whole candle beyond rs, not its close | `TL_CONFIRM_MODE = "full"` |
| a bounce should not void the setup | `TL_CONFIRM_CANCEL = False` |
| confirm 15m breaks too | add `"15m"` to `TL_CONFIRM_N` |

Eight trades over nine days is not evidence either way. `experiments/trendline_wait_search.py`
scores this across six windows against a chance baseline; that is what settles it.